IMPORTS + CONFIG

In [24]:
import ast
import json
import os
import time
from collections import Counter, defaultdict
from pathlib import Path

import nbformat
import requests
from secret import GITHUB_TOKEN
from experiments import largest_data_transformations
import numpy as np


In [25]:

# -----------------------------------
# GITHUB CONFIG
# -----------------------------------


HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
}

# Better than generic ipynb search
#QUERYS = 'extension:ipynb train_test_split OR StandardScaler'#'extension:ipynb "import pandas"'
QUERYS = [
    'extension:ipynb train_test_split',
    'extension:ipynb sklearn.preprocessing',
    'extension:ipynb LabelEncoder',
    'extension:ipynb OneHotEncoder',
    'extension:ipynb pandas.read_csv',
    'extension:ipynb RandomForestClassifier',
    'extension:ipynb XGBClassifier',
    'extension:ipynb feature engineering',
    'extension:ipynb data cleaning',
    'extension:ipynb data cleansing',
    'extension:ipynb date prep',
    'extension:ipynb exploration',
    'extension:ipynb EDA'
    ]
MAX_NOTEBOOKS_PER_QUERY = 600

GITHUB_NOTEBOOKS = 'notebooks_new'
KAGGLE_NOTEBOOKS = 'kaggle_notebooks_new'

SAVE_DIR = Path(GITHUB_NOTEBOOKS)#Path("notebooks_sanity_test")
SAVE_DIR.mkdir(exist_ok=True)

eps = 1e-10



In [26]:
transformations = list(largest_data_transformations.keys())

EXTRACT GITHUB DATA

In [27]:
# -----------------------------------
# SEARCH GITHUB NOTEBOOKS
# -----------------------------------

def search_notebooks(query, page=1):

    url = "https://api.github.com/search/code"

    params = {
        "q": query,
        "per_page": 100,
        "page": page,
    }

    r = requests.get(
        url,
        headers=HEADERS,
        params=params,
    )
    print(r.status_code)
    print(r.text)
    if r.status_code != 200:

        print("GitHub API ERROR")
        print(r.text)

        return []

    data = r.json()

    return data.get("items", [])

# -----------------------------------
# DOWNLOAD NOTEBOOK
# -----------------------------------

def github_raw_url(html_url):

    raw = html_url.replace(
        "github.com",
        "raw.githubusercontent.com"
    )

    raw = raw.replace("/blob/", "/")

    return raw


def download_notebook(item):

    raw_url = github_raw_url(
        item["html_url"]
    )

    try:

        r = requests.get(raw_url)

        if r.status_code != 200:

            print("FAILED:", raw_url)
            return False

        # verify notebook JSON

        try:

            notebook_json = r.json()

        except Exception:

            print(
                "NOT JSON:",
                raw_url
            )

            return False

        # notebook sanity check

        if "cells" not in notebook_json:

            print(
                "NO CELLS:",
                raw_url
            )

            return False

        repo_name = (
            item["repository"]["full_name"]
            .replace("/", "__")
        )

        filename = item["name"]

        out_path = (
            SAVE_DIR /
            f"{repo_name}__{filename}"
        )

        with open(
            out_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                notebook_json,
                f,
            )

        return True

    except Exception as e:

        print("ERROR:", e)

        return False

CRAWL

In [28]:
# -----------------------------------
# CRAWL NOTEBOOKS
# -----------------------------------

def crawl_notebooks():

    downloaded = 0
    page = 1
    downloaded_notebooks = set()
    for query in QUERYS:
        print("\n*************\nQUERY:", query)
        while downloaded < MAX_NOTEBOOKS_PER_QUERY:

            print(f"\nPAGE {page}")

            items = search_notebooks(query, page)

            if not items:
                print("No more results")
                break

            for item in items:
                repo_name = (
                    item["repository"]["full_name"]
                    .replace("/", "__")
                )
                filename = item["name"]
                notebook_name = f"{repo_name}__{filename}"
                if notebook_name in downloaded_notebooks:
                    print("seen this notebook!")
                    continue

                downloaded_notebooks.add(notebook_name)
                success = download_notebook(item)

                if success:

                    downloaded += 1

                    print(
                        f"Downloaded {downloaded}"
                    )

                if downloaded >= MAX_NOTEBOOKS_PER_QUERY:
                    break

                # avoid rate limits
                time.sleep(0.2)

            page += 1

        downloaded = 0
        page = 1

    print("\nDONE")

In [29]:
# -----------------------------------
# LOAD NOTEBOOK CODE CELLS
# -----------------------------------

def extract_code_cells(notebook_path):

    try:

        nb = nbformat.read(
            notebook_path,
            as_version=4,
        )

        cells = [

            c["source"]

            for c in nb.cells

            if c.cell_type == "code"
        ]

        return cells

    except Exception as e:

        print(f"FAILED: {notebook_path}")
        print(e)

        return []

SEMANTIC RULE ENGINE

In [30]:
import ast


# -----------------------------------
# SEMANTIC RULE ENGINE
# -----------------------------------

class SemanticPreprocessingVisitor(ast.NodeVisitor):

    def __init__(self):

        self.transforms = []

        # ---------------------------------
        # symbolic quartile tracking
        # ---------------------------------

        self.last_q1 = None
        self.last_q3 = None

    # -----------------------------------
    # HELPERS
    # -----------------------------------

    def resolve_constant(self, node):

        if isinstance(node, ast.Constant):
            return node.value

        return None

    def get_argument(
        self,
        node,
        kw_name,
        position,
    ):

        # keyword arg

        for kw in node.keywords:

            if kw.arg == kw_name:

                return self.resolve_constant(
                    kw.value
                )

        # positional arg

        if len(node.args) > position:

            return self.resolve_constant(
                node.args[position]
            )

        return None

    def get_raw_argument(self, node, kw_name, position):
        """Retrieves the raw AST node for an argument (keyword or positional)."""
        for kw in node.keywords:
            if kw.arg == kw_name:
                return kw.value

        if len(node.args) > position:
            return node.args[position]

        return None

    def _detect_fill_strategy(self, arg_node):
        """Walks an argument AST to detect pandas method calls (.mean, .median, .mode)

        or scikit-learn string literals ('mean', 'median', 'most_frequent').
        """
        if arg_node is None:
            return None

        for child in ast.walk(arg_node):
            # Detects pandas calls: df.fillna(df.mean()), df['col'].fillna(df['col'].median()), etc.
            if isinstance(child, ast.Call) and isinstance(child.func, ast.Attribute):
                attr = child.func.attr.lower()
                if attr == "mean":
                    return "fill_mean"
                elif attr == "median":
                    return "fill_median"
                elif attr == "mode":
                    return "fill_mode"

            # Detects string literals: SimpleImputer(strategy='mean' / 'median' / 'most_frequent')
            elif isinstance(child, ast.Constant) and isinstance(child.value, str):
                val = child.value.lower()
                if val == "mean":
                    return "fill_mean"
                elif val == "median":
                    return "fill_median"
                elif val in ["mode", "most_frequent"]:
                    return "fill_mode"

        return None

    # -----------------------------------
    # FUNCTION CALLS
    # -----------------------------------

    def visit_Call(self, node):

        # ---------------------------------
        # attribute calls
        # ---------------------------------

        if isinstance(node.func, ast.Attribute):

            attr = node.func.attr.lower()

            # qcut

            if attr == "qcut":

                q_value = self.get_argument(
                    node=node,
                    kw_name="q",
                    position=1,
                )

                if q_value in [2, 5, 10]:

                    self.transforms.append(
                        f"bin_equal_frequency_{q_value}"
                    )

            # cut

            elif attr == "cut":

                bins_value = self.get_argument(
                    node=node,
                    kw_name="bins",
                    position=1,
                )

                if bins_value in [2, 5, 10]:

                    self.transforms.append(
                        f"bin_equal_width_{bins_value}"
                    )

            # winsorize

            elif attr == "winsorize":

                self.transforms.append(
                    "winsorize"
                )

            # explicit iqr function

            elif attr == "iqr":

                self.transforms.append(
                    "IQR"
                )

            elif attr == "zscore":
                self.transforms.append(
                    "zscore"
                )

            # ---------------------------------
            # deduplication
            # ---------------------------------

            elif attr == "drop_duplicates":
                self.transforms.append(
                    "drop_duplicates"
                )

            # ---------------------------------
            # missing value operations
            # ---------------------------------

            elif attr == "dropna":
                self.transforms.append(
                    "fill_drop_na"
                )

            elif attr == "fillna":
                fill_arg = self.get_raw_argument(
                    node=node,
                    kw_name="value",
                    position=0,
                )
                strategy = self._detect_fill_strategy(fill_arg)
                if strategy:
                    self.transforms.append(strategy)

        # ---------------------------------
        # direct function calls
        # ---------------------------------

        elif isinstance(node.func, ast.Name):

            func_name = node.func.id.lower()

            # IsolationForest

            if func_name == "isolationforest":

                self.transforms.append(
                    "isolationForest"
                )

            # winsorize

            elif func_name == "winsorize":

                self.transforms.append(
                    "winsorize"
                )

            # IQR

            elif func_name == "iqr":

                self.transforms.append(
                    "IQR"
                )

            # MinMaxScaler

            elif func_name in [
                "minmaxscaler",
                "minmax_scale",
            ]:

                self.transforms.append(
                    "norm_min_max"
                )

            elif func_name == "zscore":
                self.transforms.append(
                    "zscore"
                )

            # SimpleImputer (scikit-learn)

            elif func_name == "simpleimputer":
                strategy_arg = self.get_raw_argument(
                    node=node,
                    kw_name="strategy",
                    position=0,
                )
                strategy = self._detect_fill_strategy(strategy_arg)
                if strategy:
                    self.transforms.append(strategy)

        self.generic_visit(node)

    # -----------------------------------
    # ASSIGNMENTS
    # -----------------------------------

    def visit_Assign(self, node):

        # only simple assignments

        if len(node.targets) != 1:

            self.generic_visit(node)
            return

        target = node.targets[0]

        # ---------------------------------
        # variable assignment
        # ---------------------------------

        if isinstance(target, ast.Name):

            var_name = target.id

            # ---------------------------------
            # RHS is function call
            # ---------------------------------

            if isinstance(node.value, ast.Call):

                value = node.value

                # ---------------------------------
                # attribute call
                # ---------------------------------

                if isinstance(
                    value.func,
                    ast.Attribute
                ):

                    attr = value.func.attr.lower()

                    # -------------------------
                    # quantile(.25/.75)
                    # -------------------------

                    if attr == "quantile":

                        q_value = self.get_argument(
                            node=value,
                            kw_name="q",
                            position=0,
                        )

                        # Q1

                        if q_value == 0.25:

                            self.last_q1 = var_name

                        # Q3

                        elif q_value == 0.75:

                            self.last_q3 = var_name

        # ---------------------------------
        # df["col"] = np.log(...)
        # ---------------------------------

        if isinstance(target, ast.Subscript):

            value = node.value

            if isinstance(value, ast.Call):

                if isinstance(
                    value.func,
                    ast.Attribute
                ):

                    attr = value.func.attr.lower()

                    if attr in [
                        "log",
                        "log1p",
                    ]:

                        self.transforms.append(
                            "norm_log"
                        )

        self.generic_visit(node)

    # -----------------------------------
    # BINARY OPERATIONS
    # -----------------------------------

    def visit_BinOp(self, node):

        # subtraction

        if isinstance(node.op, ast.Sub):

            left = node.left
            right = node.right

            # Q3 - Q1

            if (
                isinstance(left, ast.Name)
                and isinstance(right, ast.Name)
            ):

                left_name = left.id
                right_name = right.id

                if (
                    left_name == self.last_q3
                    and right_name == self.last_q1
                ):

                    self.transforms.append(
                        "IQR"
                    )

        self.generic_visit(node)

In [31]:
# -----------------------------------
# EXTRACT TRANSFORMS FROM CODE
# -----------------------------------

def extract_transforms_from_code(code):

    try:
        tree = ast.parse(code)

        visitor = SemanticPreprocessingVisitor()

        visitor.visit(tree)
        return visitor.transforms

    except Exception:
        return []

In [32]:
# -----------------------------------
# PROCESS SINGLE NOTEBOOK
# -----------------------------------

def process_notebook(notebook_path):

    cells = extract_code_cells(
        notebook_path
    )

    notebook_transforms = []

    for cell in cells:

        transforms = (
            extract_transforms_from_code(
                cell
            )
        )

        notebook_transforms.extend(
            transforms
        )

    return notebook_transforms

In [33]:
# -----------------------------------
# ANALYZE ALL NOTEBOOKS
# -----------------------------------

def analyze_corpus(folder_names):
    if isinstance(folder_names, (str, Path)):
            folder_names = [folder_names]

    transform_counter = Counter()
    transition_counter = defaultdict(Counter)

    # 1. Collect notebook paths from ALL folders
    notebook_paths = []
    for folder in folder_names:
        save_dir = Path(folder)
        # Extend the main list with notebooks found in this specific folder
        notebook_paths.extend(list(save_dir.rglob("*.ipynb")))

    print(f"Found {len(notebook_paths)} notebooks across {len(folder_names)} folders")

    for idx, notebook_path in enumerate(notebook_paths):
        if idx % 50 == 0:
            print(f"Processing {idx}")

        transforms = process_notebook(notebook_path)
        if transforms:
            print("\n===================")
            print(notebook_path)
            print(transforms)

        # frequency counts
        transform_counter.update(transforms)

        # transitions

        for a, b in zip(transforms[:-1], transforms[1:]):
            transition_counter[a][b] += 1

    # ---------------------------------
    # transform probabilities
    # ---------------------------------

    total = sum(transform_counter.values())

    transform_probabilities = {
        t: c / total for t, c in (transform_counter.items())
    }

    # ---------------------------------
    # transition probabilities
    # ---------------------------------

    transition_probabilities = {}

    for a, next_ops in (transition_counter.items()):
        total_transitions = sum(next_ops.values())
        transition_probabilities[a] = {
            b: c / total_transitions for b, c in (next_ops.items())
        }

    return (
        transform_probabilities,
        transition_probabilities,
    )

RUN CODE

In [34]:
# crawl_notebooks()

In [35]:
(
    transform_probabilities,
    transition_probabilities,
) = analyze_corpus(GITHUB_NOTEBOOKS)

Found 6676 notebooks across 1 folders
Processing 0

notebooks_new\10xac__crispdm-yabebalFantaye__du1.ipynb
['norm_min_max']

notebooks_new\14zip__Tubes-Data-Mining-Kelompok-2__Tubes Datmin.ipynb
['drop_duplicates', 'drop_duplicates', 'IQR', 'norm_min_max', 'fill_mean']

notebooks_new\1644heihei__Kaggle_Predicting_Loan_Payback__S5E11.ipynb
['IQR']

notebooks_new\1644heihei__Kaggle_Predicting_Stellar_Class__stellar_classification_executed.ipynb
['norm_log', 'norm_log']

notebooks_new\2026-1st__team-2__09_bumjun_data3_compact_l1_explainability.ipynb
['drop_duplicates', 'fill_mean', 'fill_mean', 'fill_median', 'fill_median', 'fill_median', 'fill_drop_na', 'bin_equal_frequency_10']

notebooks_new\21Alul21__3MTT_Machine_Learning_Capstone_project__Augustine_Alul_Agaji_Capstone_Project.ipynb
['norm_min_max']

notebooks_new\21B030720__financial-news-parsing__formatting-checkpoint.ipynb
['fill_drop_na']

notebooks_new\2303A52445__STML__SML_Ass_6.ipynb
['fill_drop_na']

notebooks_new\23eg110e35-e

<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.



notebooks_new\aakashtandon__US_options_ORATS__Spread_ROC_Arb.ipynb
['fill_drop_na']

notebooks_new\aamir-ansari-44__Study__MLFlow_Experiment.ipynb
['fill_drop_na']

notebooks_new\AaronChen007__CNS_Model_Code__Training&Testing_Aim1_FCN_LabelCorrected.ipynb
['fill_drop_na']

notebooks_new\Aathisivabalan__-AI-Based-Predictive-Modeling-for-Network-Threat-Detection__M1-DATA PREPROCESSING.ipynb
['fill_drop_na', 'drop_duplicates']

notebooks_new\AatifUsmani__Invisible-Chem-City__data_cleaning.ipynb
['fill_drop_na', 'fill_drop_na']

notebooks_new\aayandeb__Beyond-Ball-v2-Simplified-Repo__q2_nn_flat.ipynb
['fill_drop_na']

notebooks_new\AB281610__SM-API__F3.ipynb
['fill_drop_na']

notebooks_new\abawchen__kaggle-home-credit-default-risk__m_nn_10x.ipynb
['norm_min_max']

notebooks_new\abbasshahid__FinancialFraudDetection__SanaML_v2.ipynb
['fill_drop_na', 'fill_drop_na', 'fill_mode']

notebooks_new\AbdelghaffourMouhsine__Mham_AWS_Car_Parts_Scraping_Project__code.ipynb
['drop_duplicates', 'drop_du

<unknown>:39: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:45: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:47: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is


notebooks_new\AgneseNahuel__PI_ML_OPS__ML.ipynb
['drop_duplicates', 'drop_duplicates']

notebooks_new\agrwalankit942__CUSTOMER-CHURN-PREDICTION__csv.ipynb
['fill_drop_na', 'fill_median', 'fill_median']

notebooks_new\ahadxdev__Urban_Sense_AI__CA_file.ipynb
['fill_mean', 'fill_mode', 'fill_mode', 'IQR']

notebooks_new\ahadxdev__Urban_Sense_AI__TX_file.ipynb
['fill_mean', 'fill_mode', 'fill_mode', 'IQR']
Processing 250

notebooks_new\ahsanahmede7__social_network__Social_network.ipynb
['fill_median', 'fill_mode']

notebooks_new\AI-Diagnostic-Assistant__ML-Environment__UTO_XAI_classification.ipynb
['fill_drop_na']

notebooks_new\aifimmunology__ALTRA-manuscript__06.1-Python_deepclean_partition_cell_types.ipynb
['fill_drop_na']

notebooks_new\aifin-hkust__aifin-hkust.github.io__uncertainty.ipynb
['fill_median']

notebooks_new\AiniNurM__Data-Analysis-with-Python__Basket Analisis Market (1).ipynb
['zscore']

notebooks_new\aishwaryapanchal__skincare_ml__categorical-encoding.ipynb
['fill_median

...

<unknown>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.



notebooks_new\yash-dange__Credit-Score-Classification-Multi-Class-__AML_Project.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 6500

notebooks_new\Yash22222__Flask-based-Sentiment-Analysis-for-Product-Reviews__SentimentAnalysis.ipynb
['fill_drop_na']

notebooks_new\Yashr90__Titanic-survivors__Titanic.ipynb
['fill_drop_na', 'fill_median', 'fill_median', 'fill_mode']

notebooks_new\yatin-t__XRP-prediction__EDA.ipynb
['drop_duplicates', 'norm_min_max']

notebooks_new\yauheni-chekan__ML-Spring-Practical-Tasks__22_LR_JA_Yauheni_Chekan_DP.ipynb
['norm_min_max']

notebooks_new\yauheni-se__TitanicFromDisaster__TitanicFromDisaster.ipynb
['fill_drop_na', 'fill_median']

notebooks_new\YaverJavid__t_s101__ns.ipynb
['drop_duplicates', 'fill_mode'

<unknown>:12: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:35: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:36: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" i


notebooks_new\yselimd__Airbnb-Price-Prediction__data_wrangling.ipynb
['fill_median', 'fill_median', 'fill_median', 'fill_median', 'fill_median', 'fill_median', 'fill_median']

notebooks_new\Yuva2832__FMML_LAB_PROJECT__Lab_3.ipynb
['fill_drop_na']

notebooks_new\YuvrajZende__Machine-Learning-Practice__ML1.ipynb
['fill_drop_na']

notebooks_new\yuzuponikemi__machine-learning-playground__25_categorical_variable_encoding_improved_v2.ipynb
['drop_duplicates']

notebooks_new\Yvette-Ibarra__regression-project__Working_notebook_regression_zillow.ipynb
['fill_drop_na']

notebooks_new\ZaakZoeng__SSN4PaCDM__SSN4PaCDM.ipynb
['fill_drop_na', 'fill_drop_na', 'drop_duplicates', 'drop_duplicates']

notebooks_new\zafirhsn__machine-learning__.ipynb
['fill_drop_na']

notebooks_new\zahra-code__DATA-ANALYSIS-projects__TitanicDataAnalysis.ipynb
['fill_median', 'fill_median', 'fill_mode']
Processing 6600

notebooks_new\zainabnazari__ppmi__UPSIT-adaboost.ipynb
['fill_drop_na']

notebooks_new\zakaria-statistic

<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:79: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:129: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


In [36]:
# -----------------------------------
# PRINT TRANSFORM PROBABILITIES
# -----------------------------------
print("\n=== TRANSFORM PROBABILITIES ===\n")

for transform, prob in sorted(transform_probabilities.items(), key=lambda x: x[1], reverse=True):
    print(f"{transform:30s} {prob:.4f}")

# -----------------------------------
# PRINT TRANSITION PROBABILITIES
# -----------------------------------
print("\n=== TRANSITION PROBABILITIES ===\n")

for transform_a, transitions in (transition_probabilities.items()):
    print(f"\n{transform_a} ->")

    for transform_b, prob in sorted(transitions.items(), key=lambda x: x[1], reverse=True):
        print(f"    {transform_b:30s} {prob:.4f}")


=== TRANSFORM PROBABILITIES ===

fill_drop_na                   0.4052
drop_duplicates                0.1364
fill_median                    0.0948
fill_mean                      0.0934
norm_min_max                   0.0905
fill_mode                      0.0862
norm_log                       0.0363
IQR                            0.0331
zscore                         0.0096
isolationForest                0.0051
bin_equal_width_5              0.0035
bin_equal_frequency_5          0.0019
bin_equal_frequency_2          0.0016
bin_equal_frequency_10         0.0011
winsorize                      0.0008
bin_equal_width_10             0.0005

=== TRANSITION PROBABILITIES ===


drop_duplicates ->
    drop_duplicates                0.4557
    fill_drop_na                   0.2984
    IQR                            0.0623
    fill_median                    0.0525
    fill_mean                      0.0459
    norm_log                       0.0295
    fill_mode                      0.0230
    norm_

In [37]:
prob_dict = {}
for transform_op in transformations:
    prob_dict[transform_op] = transform_probabilities.get(transform_op, eps)

prob_dict['zscore_clip_3'] = transform_probabilities.get('zscore', eps)
prob_dict['zscore_filter_3'] = transform_probabilities.get('zscore', eps)
print(prob_dict)


{'fill_median': 0.09476775226908703, 'fill_mode': 0.08622530699412707, 'fill_mean': 0.09343299519487454, 'fill_drop_na': 0.405232247730913, 'bin_equal_frequency_2': 0.001601708489054992, 'bin_equal_frequency_5': 0.0018686599038974907, 'bin_equal_frequency_10': 0.0010678056593699946, 'bin_equal_width_2': 1e-10, 'bin_equal_width_5': 0.0034703683929524828, 'bin_equal_width_10': 0.0005339028296849973, 'norm_min_max': 0.09049652963160705, 'norm_log': 0.036305392418579815, 'zscore_clip_3': 0.009610250934329953, 'zscore_filter_3': 0.009610250934329953, 'winsorize': 0.000800854244527496, 'IQR': 0.03310197544046983, 'isolationForest': 0.005072076882007475, 'drop_duplicates': 0.13641217298451683}


In [38]:
import ast
import re
import pandas as pd


text = r"""

"""


# Find every line that looks like a Python list
list_strings = re.findall(r"^\[.*\]$", text, flags=re.MULTILINE)

# Convert them into actual Python lists
lists = [ast.literal_eval(s) for s in list_strings]

# print("Lists:")
# print(lists)

print(f"found {len(lists)} pipes")
# Average list size
avg_size = sum(len(lst) for lst in lists) / len(lists) if lists else 0

print(f"\nAverage list size: {avg_size:.2f}")

# Save to CSV (one row per list)
df = pd.DataFrame({
    "list": [str(lst) for lst in lists],
    "size": [len(lst) for lst in lists]
})
df.to_csv("pipelinesDataPrep.csv", index=False)

print("\nSaved to lists.csv")

found 0 pipes

Average list size: 0.00

Saved to lists.csv


KAGGLE


In [39]:
(
    transform_probabilities_kaggle,
    transition_probabilities_kaggle,
) = analyze_corpus(KAGGLE_NOTEBOOKS)

<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


Found 3900 notebooks across 1 folders
Processing 0

kaggle_notebooks_new\a0d1n0_air-quality-eda-stacked-regression-beginners.ipynb
['fill_drop_na']

kaggle_notebooks_new\aagghh_how-to-achieve-upvotes-for-a-dataset-on-kaggle.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

kaggle_notebooks_new\aakashnain_is-it-better-than-2017.ipynb
['fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na']

kaggle_notebooks_new\aakashnain_let-s-check-what-the-survey-says.ipynb
['fil

<unknown>:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.



kaggle_notebooks_new\abhashrai_customer-retention-analysis-prediction.ipynb
['fill_drop_na']

kaggle_notebooks_new\abhaymudgal_intrusion-detection-system.ipynb
['fill_drop_na', 'norm_min_max']

kaggle_notebooks_new\abhijitdahatonde_titanic-passenger-survival-prediction.ipynb
['fill_median', 'fill_median']
Processing 50

kaggle_notebooks_new\abhishek0032_data-science-toolkit-codes-skills-to-succeed.ipynb
['fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_mean', 'norm_min_max']

kaggle_notebooks_new\abhishek0032_exploring-regression-models-on-wine-data.ipynb
['fill_median', 'fill_median']

kaggle_notebooks_new\abhishek0032_titanic-survival-prediction-feature-engineering.ipynb
['fill_mean', 'fill_median', 'norm_min_max']

kaggle_notebooks_new\abhishekmamidi_time-series-analysis-artificial-neural-networks.ipynb
['norm_min_max']

kaggle_notebooks_new\abinanthank_crop-production-rice-and-wheat-abinanthan-k.ipynb
['fill_drop_na', 'norm_min_

<unknown>:17: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


...

<unknown>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.



kaggle_notebooks_new\yoohwanseol_pandas-cheatsheet-125-exercises.ipynb
['fill_mean', 'fill_mean', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

kaggle_notebooks_new\yorkyong_churn-prediction-ensemble-w-cb-xgb-lgbm.ipynb
['fill_drop_na']

kaggle_notebooks_new\yossefmohammed_true-and-fake-news-lstm-accuracy-97-90.ipynb
['drop_duplicates']

kaggle_notebooks_new\youcefbel_old-car-price-analysis-and-prediction.ipynb
['fill_drop_na']

kaggle_notebooks_new\youhanlee_stratified-sampling-for-regression-lb-1-4627.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

kaggle_notebooks_new\yousefmohamed20_titanic.ipynb
['fill_drop_na', 'norm_min_max']

kaggle_notebooks_new\youssefaboelwafa_hotel-booking-cancellation-multiple-models.ipynb
['IQR']

kaggle_notebooks_new\youssefelbadry10_credit-card-fraud-detection.ipynb
['fill_drop_na', 'fill_drop_na']

kaggle_notebooks_new\youssefelbadry10_time-series-analysis-tutorial.ipynb
['fill_drop_na']

kaggle_notebooks_new\you

<unknown>:26: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.
<unknown>:44: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.
<unknown>:55: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.



kaggle_notebooks_new\zabihullah18_car-price-prediction.ipynb
['norm_log']

kaggle_notebooks_new\zabihullah18_email-spam-detection.ipynb
['drop_duplicates']

kaggle_notebooks_new\zahidmahar_data-science-project-lifecycle-a-case-study.ipynb
['fill_drop_na']

kaggle_notebooks_new\zain280_bank-customer-churn-prediction-analysis.ipynb
['IQR', 'fill_mean', 'norm_min_max', 'fill_mode', 'fill_mode']

kaggle_notebooks_new\zain280_online-food-analysis-trends-and-insights.ipynb
['fill_mean', 'fill_mode', 'fill_mode']

kaggle_notebooks_new\zeeshanlatif_brain-tumor-segmentation-using-u-net.ipynb
['norm_min_max']

kaggle_notebooks_new\zeeshanlatif_pandas-tutorial.ipynb
['fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'drop_duplicates']

kaggle_notebooks_new\zeeshanyounas001_heart-disease-uci.ipynb
['IQR']

kaggle_notebooks_new\zephyrwang666_riiid-lgbm-bagging2-1.ipynb
['drop_duplicates']

kaggle_notebooks_new\zephyrzhan522_titanic-prediction-dl-vs-ml.ipynb
['fill_mean']

kaggle_notebooks_new\zhejin

<unknown>:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


In [40]:
prob_dict_kaggle = {}
for transform_op in transformations:
    prob_dict_kaggle[transform_op] = transform_probabilities_kaggle.get(transform_op, eps)

prob_dict_kaggle['zscore_clip_3'] = transform_probabilities_kaggle.get('zscore', eps)
prob_dict_kaggle['zscore_filter_3'] = transform_probabilities_kaggle.get('zscore', eps)
print(prob_dict_kaggle)


{'fill_median': 0.10172272354388844, 'fill_mode': 0.09372436423297785, 'fill_mean': 0.1173092698933552, 'fill_drop_na': 0.3250615258408532, 'bin_equal_frequency_2': 1e-10, 'bin_equal_frequency_5': 0.005742411812961444, 'bin_equal_frequency_10': 0.0036915504511894994, 'bin_equal_width_2': 0.00041017227235438887, 'bin_equal_width_5': 0.004511894995898278, 'bin_equal_width_10': 0.0030762920426579163, 'norm_min_max': 0.08941755537325677, 'norm_log': 0.08326497128794094, 'zscore_clip_3': 0.007178014766201805, 'zscore_filter_3': 0.007178014766201805, 'winsorize': 0.002255947497949139, 'IQR': 0.0293273174733388, 'isolationForest': 0.005332239540607055, 'drop_duplicates': 0.1279737489745693}


In [41]:
for k in prob_dict.keys():
    print(f"{k} | github: {prob_dict[k]} || kaggle: {prob_dict_kaggle[k]}")

fill_median | github: 0.09476775226908703 || kaggle: 0.10172272354388844
fill_mode | github: 0.08622530699412707 || kaggle: 0.09372436423297785
fill_mean | github: 0.09343299519487454 || kaggle: 0.1173092698933552
fill_drop_na | github: 0.405232247730913 || kaggle: 0.3250615258408532
bin_equal_frequency_2 | github: 0.001601708489054992 || kaggle: 1e-10
bin_equal_frequency_5 | github: 0.0018686599038974907 || kaggle: 0.005742411812961444
bin_equal_frequency_10 | github: 0.0010678056593699946 || kaggle: 0.0036915504511894994
bin_equal_width_2 | github: 1e-10 || kaggle: 0.00041017227235438887
bin_equal_width_5 | github: 0.0034703683929524828 || kaggle: 0.004511894995898278
bin_equal_width_10 | github: 0.0005339028296849973 || kaggle: 0.0030762920426579163
norm_min_max | github: 0.09049652963160705 || kaggle: 0.08941755537325677
norm_log | github: 0.036305392418579815 || kaggle: 0.08326497128794094
zscore_clip_3 | github: 0.009610250934329953 || kaggle: 0.007178014766201805
zscore_filter_3

In [42]:
def get_dicts_stats(dict1, dict2):
    """Calculates correlation between two dicts with the same keys."""
    # Align values by the exact same key order
    keys = list(dict1.keys())
    x = np.array([float(dict1[k]) for k in keys])
    y = np.array([float(dict2[k]) for k in keys])
    print(f"min is {min([min(x), min(y)])}, max is {max([max(x),max(y)])}")
    # Calculate and return the Pearson correlation coefficient
    corr =  np.corrcoef(x, y)[0, 1]
    mae = np.mean(np.abs(x - y))
    rmsd = np.sqrt(np.mean((x - y) ** 2))

    print(f"corr is: {corr}\nMAE is {mae}\nRMSD is: {rmsd}")

get_dicts_stats(prob_dict, prob_dict_kaggle)

min is 1e-10, max is 0.405232247730913
corr is: 0.9804971971947064
MAE is 0.010968093418526205
RMSD is: 0.02290079893197457


In [43]:
# import matplotlib.pyplot as plt
# dict1 = prob_dict
# dict2 = prob_dict_kaggle
# keys = list(dict1.keys())
# x = np.array([float(dict1[k]) for k in keys])
# y = np.array([float(dict2[k]) for k in keys])
#
# fig,ax = plt.subplots()
# ax.scatter(x,y)
# ax.set_yscale('log')
# ax.set_xscale('log')
# ax.axline((0,0),slope=1,color='grey',dashes=(3,3))
# plt.show()

In [44]:
# import matplotlib.pyplot as plt
# import numpy as np
#
# # Example data (replace with your actual probability arrays)
# method1 = np.array([float(dict1[k]) for k in keys])
# method2 = np.array([float(dict2[k]) for k in keys])
#
# # Create scatter plot
# plt.figure(figsize=(6, 6))
# plt.scatter(method1, method2, color='blue', alpha=0.7, label='Operations')
#
# # Add perfect agreement reference line (y = x)
# plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Perfect Agreement')
#
# # Formatting
# plt.xlim(0, 1)
# plt.ylim(0, 1)
# plt.xlabel('Method 1 Probabilities')
# plt.ylabel('Method 2 Probabilities')
# plt.title('Comparison Scatter Plot')
# plt.legend()
# plt.grid(True)
# plt.show()


In [45]:
import ast
import re
import pandas as pd




# Find every line that looks like a Python list
text = r"""

"""
list_strings = re.findall(r"^\[.*\]$", text, flags=re.MULTILINE)

# Convert them into actual Python lists
lists = [ast.literal_eval(s) for s in list_strings]

# print("Lists:")
# print(lists)

print(f"found {len(lists)} pipes")
# Average list size
avg_size = sum(len(lst) for lst in lists) / len(lists) if lists else 0

print(f"\nAverage list size: {avg_size:.2f}")

# Save to CSV (one row per list)
df = pd.DataFrame({
    "list": [str(lst) for lst in lists],
    "size": [len(lst) for lst in lists]
})
df.to_csv("pipelinesDataPrep_kaggle.csv", index=False)

print("\nSaved to lists.csv")

found 0 pipes

Average list size: 0.00

Saved to lists.csv


combined

In [46]:
(
    transform_probabilities_combined,
    transition_probabilities_combined,
) = analyze_corpus([KAGGLE_NOTEBOOKS, GITHUB_NOTEBOOKS])

Found 10576 notebooks across 2 folders
Processing 0

kaggle_notebooks_new\a0d1n0_air-quality-eda-stacked-regression-beginners.ipynb
['fill_drop_na']

kaggle_notebooks_new\aagghh_how-to-achieve-upvotes-for-a-dataset-on-kaggle.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

kaggle_notebooks_new\aakashnain_is-it-better-than-2017.ipynb
['fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na']

kaggle_notebooks_new\aakashnain_let-s-check-what-the-survey-says.ipynb
['fi

<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.



kaggle_notebooks_new\abdmental01_sparkle-forecast-predicting-diamond-prices.ipynb
['drop_duplicates', 'IQR', 'norm_min_max']

kaggle_notebooks_new\abdoashraf90_helthcare-diabetes-with-acuracy-99-using-knn.ipynb
['IQR']

kaggle_notebooks_new\abdrakhmanmurat_domestic-violence-in-colombia.ipynb
['fill_mode', 'fill_mode', 'fill_mode', 'fill_mode']

kaggle_notebooks_new\abhashrai_customer-retention-analysis-prediction.ipynb
['fill_drop_na']

kaggle_notebooks_new\abhaymudgal_intrusion-detection-system.ipynb
['fill_drop_na', 'norm_min_max']

kaggle_notebooks_new\abhijitdahatonde_titanic-passenger-survival-prediction.ipynb
['fill_median', 'fill_median']
Processing 50

kaggle_notebooks_new\abhishek0032_data-science-toolkit-codes-skills-to-succeed.ipynb
['fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_drop_na', 'fill_mean', 'norm_min_max']

kaggle_notebooks_new\abhishek0032_exploring-regression-models-on-wine-data.ipynb
['fill_median', 'fill_median']

kaggl

<unknown>:17: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.



kaggle_notebooks_new\adrienmorel97_eda-lightgbm-optuna-1-0644.ipynb
['bin_equal_width_10', 'fill_median']

kaggle_notebooks_new\adrienmorel97_predicting-depression-with-ensemble-learning.ipynb
['fill_median', 'norm_log', 'fill_median', 'fill_median', 'fill_mode', 'fill_median', 'isolationForest']

kaggle_notebooks_new\aeryan_spotify-music-analysis.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 100

kaggle_notebooks_new\aeshen_the-secret-to-getting-the-second-date.ipynb
['fill_drop_na']

kaggle_notebooks_new\agehsbarg_top-10-0-10943-stacking-mice-and-brutal-force.ipynb
['fill_mode', 'fill_mode', 'fill_mode', 'fill_mode', 'fill_mode', 'fill_mode', 'fill_mode', 'fill_mode', 'fill_mode', 'fill_mode', 'fill_mode', 'fill_mode', 'fill_mode']

kaggle_notebooks_new\agileteam_3rd-type1-2-3-1-2.ipynb
['fill_drop_na']

kaggle_notebooks_new\agileteam_3rd-type2-3-2-baseline.ipynb
['fill_mean', 'fill_mean', 'norm_min_max']

kaggle_notebooks_new\agileteam_insurance-starter-tutorial

...

<unknown>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.



notebooks_new\yash-dange__Credit-Score-Classification-Multi-Class-__AML_Project.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 10400

notebooks_new\Yash22222__Flask-based-Sentiment-Analysis-for-Product-Reviews__SentimentAnalysis.ipynb
['fill_drop_na']

notebooks_new\Yashr90__Titanic-survivors__Titanic.ipynb
['fill_drop_na', 'fill_median', 'fill_median', 'fill_mode']

notebooks_new\yatin-t__XRP-prediction__EDA.ipynb
['drop_duplicates', 'norm_min_max']

notebooks_new\yauheni-chekan__ML-Spring-Practical-Tasks__22_LR_JA_Yauheni_Chekan_DP.ipynb
['norm_min_max']

notebooks_new\yauheni-se__TitanicFromDisaster__TitanicFromDisaster.ipynb
['fill_drop_na', 'fill_median']

notebooks_new\YaverJavid__t_s101__ns.ipynb
['drop_duplicates', 'fill_mode

<unknown>:12: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:35: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:36: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" i


notebooks_new\zainabnazari__ppmi__UPSIT-adaboost.ipynb
['fill_drop_na']

notebooks_new\zakaria-statistics__ai-mlops__02-data-preparation.ipynb
['norm_log']

notebooks_new\ZamirPineda__spark_colab_package__Masterclass_ETL_Data_Quality.ipynb
['fill_drop_na', 'drop_duplicates']

notebooks_new\zekoNinja__Stock-Prediction__Stocks Prediction_Vale-Optimized -Copy1.ipynb
['norm_min_max']

notebooks_new\zgeblbl__perfmatch-ain311project__linear_reg_team.py-checkpoint.ipynb
['fill_drop_na', 'fill_drop_na']

notebooks_new\zhimin-z__Asset-Management-Topic-Modeling__RQ5.ipynb
['norm_log']

notebooks_new\zhuanxuhit__kaggle__1-数据处理.ipynb
['fill_mean']

notebooks_new\ZijunSong__PertDiffBench__process.ipynb
['fill_drop_na']

notebooks_new\zimkk__Anomaly-Detection-System__ADS.ipynb
['norm_log']

notebooks_new\ziyapatel0169__TEAM_BROKER_AMAZON_SALES_ANALYSES__SRC.ipynb
['fill_drop_na', 'isolationForest', 'fill_drop_na', 'fill_drop_na']
Processing 10550

notebooks_new\zleo-ai__COMP9417-Group-Project__GBM.

C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: DuplicateCellId: Non-unique cell id '7ffacb27' detected. Corrected to 'b588a429'.
  validate(nb)
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:79: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:129: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


In [47]:
prob_dict_combined = {}
for transform_op in transformations:
    prob_dict_combined[transform_op] = transform_probabilities_combined.get(transform_op, eps)

prob_dict_combined['zscore_clip_3'] = transform_probabilities_combined.get('zscore', eps)
prob_dict_combined['zscore_filter_3'] = transform_probabilities_combined.get('zscore', eps)
print(prob_dict_combined)


{'fill_median': 0.09870099744838784, 'fill_mode': 0.09046624913013222, 'fill_mean': 0.10693574576664347, 'fill_drop_na': 0.3598932962189747, 'bin_equal_frequency_2': 0.0006958942240779402, 'bin_equal_frequency_5': 0.0040593829737879845, 'bin_equal_frequency_10': 0.002551612154952447, 'bin_equal_width_2': 0.00023196474135931338, 'bin_equal_width_5': 0.0040593829737879845, 'bin_equal_width_10': 0.001971700301554164, 'norm_min_max': 0.08988633727673394, 'norm_log': 0.06286244490837392, 'zscore_clip_3': 0.008234748318255625, 'zscore_filter_3': 0.008234748318255625, 'winsorize': 0.0016237531895151936, 'IQR': 0.030967292971468337, 'isolationForest': 0.005219206680584551, 'drop_duplicates': 0.13163999072141033}
